# `<NN>` — `<method or topic>`

**Copy this file; do not edit it in place.** Save your copy as
`notebooks/NN-<short-name>.ipynb`, matching the code module and the document
subsection it belongs to.

| | |
|---|---|
| **Author** | `<your name>` |
| **Contributes to** | `xxcluster/<path>/<module>.py` |
| **Documents** | `documentation/sections/<path>/<file>.tex`, Sect. `<n.n.n>` |
| **Date** | `<date>` |

A notebook here does two jobs, and both are required:

1. **A test bed** — the place you check that what you implemented in
   `xxcluster/` behaves, before it is reported.
2. **Evidence** — the run that produced the numbers and figures in the
   document. Section 10 maps each output below to the paragraph it supports.

Keep the section order: the reviewer reads the notebook against the write-up,
and both follow the same sequence.

> **Skeleton note.** `xxcluster` currently contains contracts, not
> implementations. Cells below raise `NotImplementedError` until the
> components they call exist. That is expected; fill them in as you implement.

## 1. Setup

Seed and environment first, so everything below is reproducible and every
number can be traced to the versions that produced it (App. A).

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))   # cluster_analysis/ on the path

import numpy as np
import xxcluster
from xxcluster.evaluation.protocol import Environment, Protocol

RANDOM_STATE = 42          # the one seed; everything else derives from it
env = Environment.capture()
env

## 2. Scope

*State in two or three sentences: what is being tested, which document
subsection it evidences, and what would count as a negative result.*

Naming a negative result in advance is not a formality — it is what stops the
notebook from becoming a search for a favourable configuration.

## 3. Data

Load through `xxcluster.io`, never with a bare `read_csv`: the loader carries
the feature roles and the provenance the document requires (Sect. 3.1–3.2).

In [ ]:
from xxcluster.io.datasets import Dataset   # plus the concrete loader

dataset: Dataset = ...        # <- your loader
X = dataset.cluster_matrix()  # columns with role="cluster" only
dataset.summary()             # feeds Sect. 3.4

## 4. Protocol

The shared setup of Sect. 4.1. Take it as given — do not tune it here. A
method needing a deviation records it against itself, in its own *Application*
paragraph.

In [ ]:
protocol = Protocol(
    indices=[...],                  # validity indices, in reporting order
    n_restarts=10,
    random_state=RANDOM_STATE,
    preprocessing=...,              # shared pipeline
    n_clusters_candidates=range(2, 11),
    environment=env,
)

## 5. Contract check

Before any result, confirm the component honours the contract. A method that
fails here produces numbers that cannot be compared with anyone else's.

Once `fit` is implemented, replace the assertions with scikit-learn's
conformance suite:

```python
from sklearn.utils.estimator_checks import check_estimator
check_estimator(Method(n_clusters=3))
```

In [ ]:
from sklearn.base import clone

method = ...                     # your component, e.g. REGISTRY.create("kmeans", n_clusters=3)

assert clone(method).get_params() == method.get_params(), "params must round-trip"
assert not hasattr(method, "labels_"), "no fitted state before fit()"

method.fit(X)

assert hasattr(method, "labels_") and hasattr(method, "n_clusters_")
assert method.labels_.shape[0] == X.shape[0]
print(method._capabilities)      # must match what the write-up claims

## 6. Selection

How \|C\| — or the density parameters — were chosen, and whether the result
survives perturbation. Both, not either: an optimal partition that is
unstable is not a finding (Sect. 4.3).

In [ ]:
from xxcluster.selection.stability import StabilityAnalysis

selector = ...                   # e.g. a BaseNClustersSelector over protocol.n_clusters_candidates
selector.fit(X)
print(selector.best_params_, "conclusive:", getattr(selector, "conclusive_", None))

stability = StabilityAnalysis(estimator=method, n_repeats=10, random_state=RANDOM_STATE)
stability.fit(X)
print(f"stability {stability.stability_:.3f} +/- {stability.stability_std_:.3f}")

## 7. Results

Every index in `protocol.indices`, including the unfavourable ones. Reporting
only the flattering subset is the failure mode this structure exists to
prevent.

In [ ]:
from xxcluster.evaluation.report import ComparisonRun, profile_clusters

run = ComparisonRun(methods=[method], protocol=protocol)
results = run.run(X)
results

In [ ]:
# Clusters in original units -- the input to naming a regime (Sect. 4.4).
profiles = profile_clusters(dataset.X, method.labels_)
profiles

## 8. Figures

Save into `documentation/figures/` under the name the `\\includegraphics` line
uses, so the document picks up a regenerated figure without being edited.

In [ ]:
import matplotlib.pyplot as plt
from xxcluster.viz import diagnostics

FIGURES = Path("../documentation/figures")

ax = diagnostics.plot_selection_curve(selector.curve_, selected=selector.best_params_)
plt.savefig(FIGURES / "<name>-selection.png", dpi=200, bbox_inches="tight")

## 9. Export

Write the tables the document `\\input`s, and store the artefact with its
protocol and environment. Numbers reach the document this way and no other:
a retyped number is the most likely place for a result to be corrupted.

In [ ]:
from xxcluster.evaluation.report import ComparisonTable

table = ComparisonTable(results)
table.to_csv("../documentation/tables/<name>-results.csv")
table.to_latex("../documentation/tables/<name>-results.tex", label="tab:<name>:results")

## 10. Findings and caveats

Fill this in last. It is the bridge to the write-up: each row names an output
above and the paragraph it supports.

| Output | Goes into | Finding |
|---|---|---|
| Section 7 scores | *Results* | `<one sentence>` |
| Section 7 profiles | *Application to AquaBlend data* / naming | `<one sentence>` |
| Section 6 curve and stability | *Hyperparameters and tuning* | `<one sentence>` |
| Section 8 figures | *Results* | `<figure numbers>` |
| Anything that failed | *Limitations* | `<one sentence>` |

**Caveats.** Inconclusive selections, unstable partitions, failed runs, and
deviations from the protocol. These belong in the document's *Limitations*
paragraph and in Sect. 4.5 — they are part of the result, not an embarrassment
to be omitted.

## Before submitting

- [ ] Runs top to bottom on a restarted kernel — *Kernel → Restart & Run All*.
- [ ] `RANDOM_STATE` set once and used everywhere; no unseeded randomness.
- [ ] Environment captured in Section 1 and stored with the artefact.
- [ ] Protocol taken from the shared setup; any deviation stated and justified.
- [ ] Contract check passes.
- [ ] Every index in `protocol.indices` reported, favourable or not.
- [ ] Figures saved under the names the document references.
- [ ] Tables exported, not retyped.
- [ ] Section 10 completed, caveats included.
- [ ] Outputs left in the committed file — they are the evidence.
- [ ] AI assistance declared per Deakin policy if used.